# EDA relacional e interactivo — revisión de visualización — E & M D01/D02 V5
**· Exploración de 24 entidades y sus relaciones**

Carga el Excel, conviértelo en CSV independientes y explora gráficos dinámicos: viajes, puntos de entrega, demanda, puntualidad, demoras, descarga, capacidad, vehículos, distancias, correlaciones e integridad. Incluye indicadores **mínimo · P25 · mediana · promedio · P75 · P95 · máximo**.

**Metodología:** el histórico V5 contiene información operacional reconstruida y completada; estos análisis describen el dataset experimental, no certifican que todas sus variables sean observaciones GPS ni prueban eficacia MARL.


## 0. Preparación y lectura del archivo

In [ ]:
%pip -q install pandas openpyxl plotly xlsxwriter ipywidgets


In [ ]:
import io, os, re, math, warnings, collections
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML
warnings.filterwarnings('ignore', category=FutureWarning)
px.defaults.template = 'plotly_white'
PALETTE = ['#2563eb','#0f766e','#f59e0b','#dc2626','#7c3aed','#0891b2','#64748b']
PLOTLY_FIGS = []
def mostrar(fig, titulo=None):
    if titulo: fig.update_layout(title={'text':titulo,'x':0.03})
    fig.update_layout(font=dict(family='Arial',size=12), margin=dict(l=35,r=25,t=65,b=35),
                      paper_bgcolor='white', plot_bgcolor='white',height=440)
    fig.show(); PLOTLY_FIGS.append(fig)

def panel(titulo, objeto):
    print('\n'+'═'*78+'\n'+titulo+'\n'+'═'*78);display(objeto)

def numero(x):
    return f'{x:,.0f}' if pd.notna(x) else '—'

In [ ]:
# CARGA: recomendado subir el Excel cuando lo solicite la celda.
# Alternativamente: RUTA_DRIVE = '/content/drive/MyDrive/mi_carpeta/archivo.xlsx'
RUTA_DRIVE = ''
if RUTA_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    excel_path = RUTA_DRIVE
else:
    from google.colab import files
    print('Selecciona tu Excel de datos EYM D01/D02 (V5):')
    subidos = files.upload()
    if not subidos: raise ValueError('No se ha cargado ningún archivo Excel.')
    excel_path = next(iter(subidos))
assert os.path.isfile(excel_path), f'No existe: {excel_path}'
print('Archivo de entrada:', os.path.basename(excel_path), '| MB:',round(os.path.getsize(excel_path)/1e6,2))


Selecciona tu Excel de datos EYM D01/D02 (V5):


Saving data_eym_d01-d02-v5.xlsx to data_eym_d01-d02-v5.xlsx
Archivo de entrada: data_eym_d01-d02-v5.xlsx | MB: 1.48


In [ ]:
# Leer TODAS las hojas sin interpretar automáticamente fechas (hay columnas con nombres heterogéneos).
libro = pd.read_excel(excel_path, sheet_name=None, engine='openpyxl')
datos = {k:v.copy() for k,v in libro.items() if not k.upper().startswith('ET22_') and not k.upper().startswith('ET23_')}
auditorias_previas = {k:v for k,v in libro.items() if k not in datos}
for nombre,df in datos.items():
    df.columns = [str(c).strip() for c in df.columns]
print('Hojas operativas:',len(datos),'| Hojas ET históricas:',len(auditorias_previas))
print('Operativas:',', '.join(datos))

def tabla(nombre):
    return datos.get(nombre, pd.DataFrame()).copy()

def numeric(df, cols):
    for c in cols:
        if c in df:df[c]=pd.to_numeric(df[c],errors='coerce')
    return df

def fechas(df,cols):
    for c in cols:
        if c in df: df[c]=pd.to_datetime(df[c],errors='coerce')
    return df

viaje = tabla('viaje');entrega=tabla('entrega');parada=tabla('parada');recorrido=tabla('recorrido')
solicitud=tabla('solicitud');asignacion=tabla('asignacion');vehiculo=tabla('vehiculo')
conductor=tabla('conductor');punto=tabla('punto_entrega');detalle=tabla('detalle_entrega')
movimiento=tabla('movimiento_carga');jornada=tabla('jornada_conductor');gasto=tabla('gasto_viaje')
for d in [viaje,entrega,parada,recorrido,solicitud,asignacion,vehiculo,conductor,punto,detalle,movimiento,jornada,gasto]:
    for c in d.columns:
        if c.startswith('id_') or c in ['secuencia','secuencia_plan','secuencia_real','cantidad','cantidad_plan','cantidad_real','bultos','bultos_plan','bultos_real','total_bultos','carga_max_kg','peso_total_kg','peso_kg','peso_real_kg','distancia_km','importe','costo_plan','costo_real','conduccion_acumulada_min','descanso_acumulado_min','latitud','longitud']:
            d[c]=pd.to_numeric(d[c],errors='coerce')
    for c in d.columns:
        if any(x in c for x in ['fecha_','_plan','_real','inicio_','fin_','llegada_','salida_','entrega_desde','entrega_hasta','entrega_objetivo','recojo_']) and not c.startswith(('id_','peso_','costo_','secuencia_')):
            if d[c].dtype=='object' or pd.api.types.is_datetime64_any_dtype(d[c]):
                dt=pd.to_datetime(d[c],errors='coerce')
                if dt.notna().sum() >= max(1,int(.4*d[c].notna().sum())): d[c]=dt
# IMPORTANTE: fechas de inicio/fin en jornadas también se convierten explícitamente
fechas(jornada,['inicio','fin'])


Hojas operativas: 24 | Hojas ET históricas: 9
Operativas: empresa, cliente, conductor, vehiculo, tipo_carga, presentacion_carga, punto_entrega, solicitud, detalle_carga, viaje, asignacion, carga_viaje, entrega, detalle_entrega, movimiento_carga, parada, recorrido, posicion, gasto_viaje, precio_combustible, tarifa_operativa, jornada_conductor, regla_conduccion_descanso, punto_descanso


,id_jornada,id_conductor,inicio,fin,conduccion_acumulada_min,conduccion_continua_min,descanso_acumulado_min,ultimo_descanso_fin,estado_jornada,id_regla_descanso
0,200001,5,2025-12-07 08:18:00,2025-12-11 15:09:00,2092,NaN,2340,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
1,200002,6,2025-12-07 08:03:00,2025-12-09 03:25:00,791,NaN,180,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
2,200003,7,2025-12-07 08:39:00,2025-12-11 13:19:00,2045,NaN,2340,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
3,200004,9,2025-12-07 08:06:00,2025-12-11 15:09:00,2014,NaN,1680,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
4,200005,10,2025-12-07 16:59:00,2025-12-09 06:45:00,813,NaN,180,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
...,...,...,...,...,...,...,...,...,...,...
495,200496,5,2026-08-01 11:11:00,2026-08-04 06:25:00,1509,NaN,930,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
496,200497,6,2026-08-02 13:31:00,2026-08-04 04:51:00,783,NaN,180,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
497,200498,7,2026-08-03 19:00:00,2026-08-05 06:55:00,814,NaN,180,NaN,FINALIZADA_PENDIENTE_VALIDACION,1
498,200499,9,2026-08-16 08:23:00,2026-08-19 22:50:00,1555,NaN,1590,NaN,FINALIZADA_PENDIENTE_VALIDACION,1


## 0.1. Conversión Excel → CSV (una tabla por entidad)

El Excel se conserva intacto. Generamos un CSV UTF-8 por hoja operativa, un manifiesto con cantidad de filas y columnas y un ZIP descargable. **No unir todas las hojas en un solo CSV**: son entidades relacionadas mediante claves.

A continuación el notebook vuelve a cargar los CSV como DataFrames para que el resto del análisis se ejecute sobre los archivos convertidos.


In [ ]:
import zipfile, shutil
from pathlib import Path
CARPETA_CSV = Path('/content/eym_csv') if Path('/content').exists() else Path('eym_csv')
CARPETA_CSV.mkdir(parents=True, exist_ok=True)
manifest=[]
for nombre, df in datos.items():
    archivo = CARPETA_CSV / f'{nombre}.csv'
    df.to_csv(archivo, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d %H:%M:%S')
    manifest.append({'entidad':nombre,'filas':len(df),'columnas':len(df.columns),'archivo':archivo.name})
manifest=pd.DataFrame(manifest)
manifest.to_csv(CARPETA_CSV/'MANIFIESTO.csv',index=False,encoding='utf-8-sig')
RUTA_ZIP=str(CARPETA_CSV.parent/'EYM_D01_D02_CSV.zip')
with zipfile.ZipFile(RUTA_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in sorted(CARPETA_CSV.glob('*.csv')): z.write(archivo,arcname=archivo.name)
# Lectura normal desde los CSV, conservando el resto de las rutinas del notebook.
datos={p.stem:pd.read_csv(p,encoding='utf-8-sig',low_memory=False) for p in CARPETA_CSV.glob('*.csv') if p.stem!='MANIFIESTO'}
# Reconstruir tipos clave (el CSV por definición no conserva el tipo fecha).
for nombre,df in datos.items():
    for c in df.columns:
        if c.startswith('id_') or c in ['secuencia','secuencia_plan','secuencia_real','total_bultos','peso_total_kg','peso_kg','distancia_km','carga_max_kg','latitud','longitud','importe','costo_plan','costo_real']:
            df[c]=pd.to_numeric(df[c],errors='coerce')
        elif any(x in c for x in ['fecha_','_inicio','_fin','llegada_','salida_','entrega_desde','entrega_hasta','entrega_objetivo','recojo_']) and (df[c].dtype=='object'):
            serie=pd.to_datetime(df[c],errors='coerce')
            if serie.notna().sum() >= max(1,int(df[c].notna().sum()*.40)):df[c]=serie
viaje=tabla('viaje');entrega=tabla('entrega');parada=tabla('parada');recorrido=tabla('recorrido')
solicitud=tabla('solicitud');asignacion=tabla('asignacion');vehiculo=tabla('vehiculo')
conductor=tabla('conductor');punto=tabla('punto_entrega');detalle=tabla('detalle_entrega')
movimiento=tabla('movimiento_carga');jornada=tabla('jornada_conductor');gasto=tabla('gasto_viaje')
fechas(jornada,['inicio','fin'])
display(manifest)
print(f'CSV operativos generados: {len(manifest)} | ZIP: {RUTA_ZIP}')


,entidad,filas,columnas,archivo
0,empresa,1,5,empresa.csv
1,cliente,8,6,cliente.csv
2,conductor,9,6,conductor.csv
3,vehiculo,9,25,vehiculo.csv
4,tipo_carga,1,15,tipo_carga.csv
5,presentacion_carga,1,14,presentacion_carga.csv
6,punto_entrega,25,15,punto_entrega.csv
7,solicitud,500,13,solicitud.csv
8,detalle_carga,500,13,detalle_carga.csv
9,viaje,500,29,viaje.csv


CSV operativos generados: 24 | ZIP: /content/EYM_D01_D02_CSV.zip


## 1. Panorama general del conjunto de datos

In [ ]:
resumen_hojas=pd.DataFrame([{'hoja':k,'registros':len(v),'columnas':len(v.columns),
        'celdas_nulas':int(v.isna().sum().sum()),
        'porc_nulos':round(100*v.isna().sum().sum()/max(1,v.size),1)} for k,v in datos.items()])
panel('Inventario de hojas D01 / D02',resumen_hojas.sort_values('registros',ascending=False).reset_index(drop=True))
mostrar(px.bar(resumen_hojas.sort_values('registros',ascending=True),x='registros',y='hoja',orientation='h',
  color='registros',color_continuous_scale='Blues',text='registros'), 'Número de registros por entidad')



══════════════════════════════════════════════════════════════════════════════
Inventario de hojas D01 / D02
══════════════════════════════════════════════════════════════════════════════


,hoja,registros,columnas,celdas_nulas,porc_nulos
0,detalle_entrega,2162,13,4324,15.4
1,parada,1982,17,9910,29.4
2,movimiento_carga,1982,12,0,0.0
3,entrega,1982,25,3964,8.0
4,recorrido,1982,13,7928,30.8
5,solicitud,500,13,1000,15.4
6,asignacion,500,9,0,0.0
7,carga_viaje,500,11,500,9.1
8,detalle_carga,500,13,500,7.7
9,jornada_conductor,500,10,1000,20.0


In [ ]:
indicadores = {'Viajes':len(viaje),'Solicitudes':len(solicitud),'Vehículos':len(vehiculo),'Conductores':len(conductor),
'Puntos de entrega':len(punto),'Entregas':len(entrega),'Paradas':len(parada),'Recorridos':len(recorrido)}
fig=go.Figure();fig.add_trace(go.Bar(x=list(indicadores),y=list(indicadores.values()),marker_color=PALETTE[0],text=list(indicadores.values()),textposition='outside'))
mostrar(fig,'Volumen por entidad principal')


## 2. Calidad y completitud de datos

In [ ]:
calidad=[]
for nom,df in datos.items():
 for campo in df.columns:
  calidad.append({'hoja':nom,'campo':campo,'registros':len(df),'faltantes':int(df[campo].isna().sum()),
    'pct_faltante':round(100*df[campo].isna().mean(),2) if len(df) else np.nan,
    'distintos':int(df[campo].nunique(dropna=True))})
calidad=pd.DataFrame(calidad)
panel('Campos con ausencia de datos (top 35)',calidad.loc[calidad.faltantes>0].sort_values(['pct_faltante','faltantes'],ascending=False).head(35))
mostrar(px.bar(resumen_hojas.sort_values('porc_nulos'),x='porc_nulos',y='hoja',orientation='h',
color='porc_nulos',color_continuous_scale='Oranges',text='porc_nulos'), 'Porcentaje de celdas sin dato por hoja')



══════════════════════════════════════════════════════════════════════════════
Campos con ausencia de datos (top 35)
══════════════════════════════════════════════════════════════════════════════


,hoja,campo,registros,faltantes,pct_faltante,distintos
26,detalle_entrega,motivo_dif,2162,2162,100.0,0
29,detalle_entrega,vol_real_m3,2162,2162,100.0,0
45,parada,id_lugar,1982,1982,100.0,0
54,parada,latitud,1982,1982,100.0,0
55,parada,longitud,1982,1982,100.0,0
56,parada,motivo,1982,1982,100.0,0
57,parada,costo,1982,1982,100.0,0
118,entrega,penalidad_hora,1982,1982,100.0,0
124,entrega,tolerancia_tardanza_min,1982,1982,100.0,0
136,recorrido,id_tramo,1982,1982,100.0,0


In [ ]:
# Chequeo de integridad básica, sin confundir ausencia en columnas opcionales con error.
PK = {'empresa':'id_empresa','cliente':'id_cliente','conductor':'id_conductor','vehiculo':'id_vehiculo',
 'punto_entrega':'id_punto','solicitud':'id_solicitud','detalle_carga':'id_detalle','viaje':'id_viaje',
 'asignacion':'id_asignacion','carga_viaje':'id_carga_viaje','entrega':'id_entrega','detalle_entrega':'id_detalle_entrega',
 'movimiento_carga':'id_movimiento','parada':'id_parada','recorrido':'id_recorrido','gasto_viaje':'id_gasto',
 'jornada_conductor':'id_jornada'}
controles=[]
for nom,pk in PK.items():
 df=tabla(nom)
 if pk in df:
  controles.append({'entidad':nom,'campo_PK':pk,'nulos_PK':int(df[pk].isna().sum()),
      'duplicados_PK':int(df[pk].dropna().duplicated().sum())})
panel('Unicidad de claves primarias',pd.DataFrame(controles))

# Validar relaciones reales disponibles según nombre del campo
FK=[('viaje','id_vehiculo','vehiculo','id_vehiculo'),('viaje','id_conductor','conductor','id_conductor'),
('asignacion','id_viaje','viaje','id_viaje'),('asignacion','id_solicitud','solicitud','id_solicitud'),
('entrega','id_asignacion','asignacion','id_asignacion'),('entrega','id_punto','punto_entrega','id_punto'),
('parada','id_viaje','viaje','id_viaje'),('parada','id_entrega','entrega','id_entrega'),
('recorrido','id_viaje','viaje','id_viaje'),('detalle_entrega','id_entrega','entrega','id_entrega'),
('movimiento_carga','id_viaje','viaje','id_viaje')]
resultados_fk=[]
for tabla_origen,col,tabla_destino,col_destino in FK:
 a=tabla(tabla_origen);target=tabla(tabla_destino)
 if col in a and col_destino in target:
  valores=pd.to_numeric(a[col],errors='coerce');ref=set(pd.to_numeric(target[col_destino],errors='coerce').dropna())
  resultados_fk.append({'relacion':f'{tabla_origen}.{col} → {tabla_destino}.{col_destino}',
  'referencias_no_nulas':int(valores.notna().sum()),'sin_correspondencia':int((valores.notna() & ~valores.isin(ref)).sum())})
panel('Integridad referencial (identificadores sin correspondencia)',pd.DataFrame(resultados_fk))



══════════════════════════════════════════════════════════════════════════════
Unicidad de claves primarias
══════════════════════════════════════════════════════════════════════════════


,entidad,campo_PK,nulos_PK,duplicados_PK
0,empresa,id_empresa,0,0
1,cliente,id_cliente,0,0
2,conductor,id_conductor,0,0
3,vehiculo,id_vehiculo,0,0
4,punto_entrega,id_punto,0,0
5,solicitud,id_solicitud,0,0
6,detalle_carga,id_detalle,0,0
7,viaje,id_viaje,0,0
8,asignacion,id_asignacion,0,0
9,carga_viaje,id_carga_viaje,0,0



══════════════════════════════════════════════════════════════════════════════
Integridad referencial (identificadores sin correspondencia)
══════════════════════════════════════════════════════════════════════════════


,relacion,referencias_no_nulas,sin_correspondencia
0,viaje.id_vehiculo → vehiculo.id_vehiculo,500,0
1,viaje.id_conductor → conductor.id_conductor,500,0
2,asignacion.id_viaje → viaje.id_viaje,500,0
3,asignacion.id_solicitud → solicitud.id_solicitud,500,0
4,entrega.id_asignacion → asignacion.id_asignacion,1982,0
5,entrega.id_punto → punto_entrega.id_punto,1982,0
6,parada.id_viaje → viaje.id_viaje,1982,0
7,parada.id_entrega → entrega.id_entrega,1982,0
8,recorrido.id_viaje → viaje.id_viaje,1982,0
9,detalle_entrega.id_entrega → entrega.id_entrega,2162,0


## 3. Evolución temporal y estados de los viajes

In [ ]:
display(Markdown('**Columnas de `viaje`**: '+', '.join(map(lambda x:'`'+x+'`',viaje.columns))))
if 'fin_carga_real' in viaje:
 viaje['fecha_inicio_eda']=pd.to_datetime(viaje['fin_carga_real'],errors='coerce')
elif 'llegada_carga_real' in viaje:
 viaje['fecha_inicio_eda']=pd.to_datetime(viaje['llegada_carga_real'],errors='coerce')
else: viaje['fecha_inicio_eda']=pd.NaT
viaje['mes_eda']=viaje['fecha_inicio_eda'].dt.to_period('M').astype(str)
mostrar(px.bar(viaje.groupby('mes_eda',as_index=False).size(),x='mes_eda',y='size',text='size',color_discrete_sequence=[PALETTE[0]]),
       'Viajes por mes de inicio')
if 'estado' in viaje:
 mostrar(px.pie(viaje,names='estado',hole=.48,color_discrete_sequence=PALETTE),'Distribución de estados de viaje')


**Columnas de `viaje`**: `id_viaje`, `codigo`, `id_vehiculo`, `id_conductor`, `id_origen`, `id_destino`, `llegada_carga_plan`, `llegada_carga_real`, `inicio_carga_plan`, `inicio_carga_real`, `fin_carga_plan`, `fin_carga_real`, `inicio_viaje_plan`, `inicio_viaje_real`, `fin_viaje_plan`, `fin_viaje_real`, `inicio_descarga_plan`, `inicio_descarga_real`, `fin_descarga_plan`, `fin_descarga_real`, `carga_inicial_kg`, `bultos_iniciales`, `distancia_plan_km`, `distancia_real_km`, `costo_plan`, `costo_real`, `estado`, `volumen_ini_m3`, `pallets_ini`

## 4. Demanda, cajas, pesos y capacidad de vehículos

In [ ]:
# Viaje -> asignación -> solicitud para carga de cajas y peso
base_viajes=viaje[['id_viaje','id_vehiculo','id_conductor','fecha_inicio_eda','mes_eda']].copy()
if {'id_viaje','id_solicitud'}.issubset(asignacion.columns) and {'id_solicitud','total_bultos','peso_total_kg'}.issubset(solicitud.columns):
 dem=asignacion[['id_viaje','id_solicitud']].merge(solicitud[['id_solicitud','total_bultos','peso_total_kg']],on='id_solicitud',how='left')
 dem=dem.groupby('id_viaje',as_index=False)[['total_bultos','peso_total_kg']].sum(min_count=1)
 base_viajes=base_viajes.merge(dem,on='id_viaje',how='left')
if 'carga_max_kg' in vehiculo:
 base_viajes=base_viajes.merge(vehiculo[['id_vehiculo','carga_max_kg']].drop_duplicates('id_vehiculo'),on='id_vehiculo',how='left')
 base_viajes['ocupacion_pct']=100*base_viajes['peso_total_kg']/base_viajes['carga_max_kg'].replace(0,np.nan)
panel('Estadísticos descriptivos de demanda',base_viajes[['total_bultos','peso_total_kg']+(['ocupacion_pct'] if 'ocupacion_pct' in base_viajes else [])].describe(percentiles=[.1,.25,.5,.75,.9,.95]).T.round(2))
mostrar(px.histogram(base_viajes,x='total_bultos',nbins=35,marginal='box',color_discrete_sequence=[PALETTE[0]]),'Distribución de cajas por viaje')
mostrar(px.histogram(base_viajes,x='peso_total_kg',nbins=35,marginal='box',color_discrete_sequence=[PALETTE[1]]),'Distribución de peso total por viaje (kg)')
if 'ocupacion_pct' in base_viajes:
 mostrar(px.histogram(base_viajes,x='ocupacion_pct',nbins=30,color_discrete_sequence=[PALETTE[2]]),'Ocupación de capacidad de carga por viaje (%)')
 panel('Viajes que superarían la capacidad declarada',base_viajes.loc[base_viajes.ocupacion_pct>100,['id_viaje','id_vehiculo','peso_total_kg','carga_max_kg','ocupacion_pct']].sort_values('ocupacion_pct',ascending=False).head(30))



══════════════════════════════════════════════════════════════════════════════
Estadísticos descriptivos de demanda
══════════════════════════════════════════════════════════════════════════════


,count,mean,std,min,10%,25%,50%,75%,90%,95%,max
total_bultos,500.0,933.06,250.42,400.00,505.80,871.00,974.50,1105.00,1182.00,1246.00,1589.00
peso_total_kg,500.0,8974.22,2659.42,3315.38,4905.39,6989.67,9388.94,10767.58,12222.48,12922.16,15681.55
ocupacion_pct,500.0,69.45,20.66,20.72,36.62,57.38,71.32,83.81,98.60,99.80,99.99



══════════════════════════════════════════════════════════════════════════════
Viajes que superarían la capacidad declarada
══════════════════════════════════════════════════════════════════════════════


,id_viaje,id_vehiculo,peso_total_kg,carga_max_kg,ocupacion_pct


## 5. Destinos, demanda geográfica y mapa de entregas

In [ ]:
# Usamos claves verificadas, no inferimos nombres de campos ausentes.
puntos_cols=list(punto.columns);display(Markdown('Campos en `punto_entrega`: '+', '.join('`'+x+'`' for x in puntos_cols)))
dest=entrega.groupby('id_punto',as_index=False).agg(entregas=('id_entrega','count')) if {'id_punto','id_entrega'}.issubset(entrega) else pd.DataFrame()
if not dest.empty:
 if {'id_punto','nombre'}.issubset(punto): dest=dest.merge(punto[['id_punto','nombre']],on='id_punto',how='left')
 dest['etiqueta']=dest['id_punto'].astype('Int64').astype(str)+' | '+dest.get('nombre',pd.Series(['Punto']*len(dest))).fillna('Punto').astype(str).str.slice(0,35)
 mostrar(px.bar(dest.sort_values('entregas').tail(25),x='entregas',y='etiqueta',orientation='h',color='entregas',color_continuous_scale='Teal'), 'Entregas por punto de destino')
 # lat/lon del Excel tal como figuran en el archivo
 if {'latitud','longitud'}.issubset(punto):
  mp=punto[['id_punto','latitud','longitud']].merge(dest[['id_punto','entregas']],on='id_punto',how='left')
  mp=mp.dropna(subset=['latitud','longitud']);mp=mp[mp.latitud.between(-19,0)&mp.longitud.between(-82,-68)]
  if not mp.empty:
   mp['entregas']=pd.to_numeric(mp['entregas'],errors='coerce').fillna(0)
   display(Markdown('**Mapa interactivo:** Carto-Positron es un proveedor externo. Si Colab no puede cargar sus teselas, el gráfico de coordenadas situado debajo continúa funcionando sin conexión.'))
   try:
    fig=px.scatter_map(mp,lat='latitud',lon='longitud',size='entregas',color='entregas',
          hover_name='id_punto',zoom=4,center={'lat':-9.5,'lon':-77.2},
          map_style='carto-positron',color_continuous_scale='Viridis',
          title='Puntos de entrega — mapa de referencia (Carto)')
    mostrar(fig)
   except Exception as exc:
    print('No se pudo generar el mapa con teselas externas:',exc)
   fig_geo=px.scatter(mp,x='longitud',y='latitud',size='entregas',color='entregas',
           hover_name='id_punto',size_max=30,color_continuous_scale='Viridis',
           title='Puntos de entrega — coordenadas (sin proveedor cartográfico)',
           labels={'longitud':'Longitud','latitud':'Latitud'})
   fig_geo.update_yaxes(scaleanchor='x',scaleratio=1)
   mostrar(fig_geo)
   repetidos=(mp.groupby(['latitud','longitud'],dropna=False)
                 .agg(puntos=('id_punto','nunique'),entregas=('entregas','sum'))
                 .query('puntos > 1').reset_index())
   if len(repetidos):panel('Coordenadas compartidas por varios puntos (verificar direcciones)',repetidos)


Campos en `punto_entrega`: `id_punto`, `id_cliente`, `id_lugar`, `codigo`, `nombre`, `contacto`, `hora_apertura`, `hora_cierre`, `tiempo_descarga`, `permite_pesados`, `restricciones`, `estado`, `capacidad_atencion_simultanea`, `latitud`, `longitud`

**Mapa interactivo:** Carto-Positron es un proveedor externo. Si Colab no puede cargar sus teselas, el gráfico de coordenadas situado debajo continúa funcionando sin conexión.


══════════════════════════════════════════════════════════════════════════════
Coordenadas compartidas por varios puntos (verificar direcciones)
══════════════════════════════════════════════════════════════════════════════


,latitud,longitud,puntos,entregas
0,-12.0681,-75.2101,2,128
1,-6.8238,-79.8611,2,121
2,-6.7714,-79.8409,2,128
3,-5.1819,-80.6572,2,172
4,-5.0925,-80.1625,2,96


## 6. Entregas: puntualidad, ventanas y tiempos de descarga

In [ ]:
display(Markdown('**Campos `entrega`:** '+', '.join('`'+x+'`' for x in entrega.columns)))
for c in ['entrega_desde','entrega_hasta','llegada_real','inicio_descarga','fin_descarga','salida_real','entrega_objetivo']:
 if c in entrega: entrega[c]=pd.to_datetime(entrega[c],errors='coerce')
if {'fin_descarga','entrega_hasta'}.issubset(entrega):
 entrega['tardanza_ventana_min']=((entrega['fin_descarga']-entrega['entrega_hasta']).dt.total_seconds()/60).clip(lower=0)
 entrega['puntual_ventana_eda']=np.where(entrega['tardanza_ventana_min'].isna(),'SIN_DATO',np.where(entrega['tardanza_ventana_min']<=0,'A_TIEMPO','FUERA_DE_VENTANA'))
 panel('Puntualidad de descarga respecto del cierre de ventana (sin tolerancia)',entrega['puntual_ventana_eda'].value_counts(dropna=False).rename_axis('clasificación').reset_index(name='entregas'))
 mostrar(px.pie(entrega,names='puntual_ventana_eda',hole=.52,color_discrete_sequence=[PALETTE[1],PALETTE[3],PALETTE[6]]),'Cumplimiento de ventana: fin de descarga')
 mostrar(px.histogram(entrega.loc[entrega.tardanza_ventana_min>0],x='tardanza_ventana_min',nbins=35,color_discrete_sequence=[PALETTE[3]]), 'Minutos de tardanza entre entregas fuera de ventana')
if {'inicio_descarga','fin_descarga'}.issubset(entrega):
 entrega['duracion_descarga_min']=(entrega.fin_descarga-entrega.inicio_descarga).dt.total_seconds()/60
 mostrar(px.histogram(entrega,x='duracion_descarga_min',nbins=40,marginal='box',color_discrete_sequence=[PALETTE[0]]),'Duración de descarga por entrega (min)')
 panel('Duraciones de descarga negativas o superiores a 8 horas (revisión)', entrega.loc[(entrega.duracion_descarga_min<0)|(entrega.duracion_descarga_min>480),['id_entrega','id_asignacion','duracion_descarga_min']].head(25))
if {'llegada_real','entrega_objetivo'}.issubset(entrega):
 entrega['desviacion_llegada_plan_min']=(entrega.llegada_real-entrega.entrega_objetivo).dt.total_seconds()/60
 mostrar(px.histogram(entrega,x='desviacion_llegada_plan_min',nbins=60,color_discrete_sequence=[PALETTE[4]]),'Desviación de llegada respecto a objetivo (min; signo positivo = tardanza)')


**Campos `entrega`:** `id_entrega`, `id_asignacion`, `id_punto`, `secuencia_plan`, `secuencia_real`, `entrega_desde`, `entrega_hasta`, `entrega_objetivo`, `llegada_real`, `inicio_descarga`, `fin_descarga`, `salida_real`, `peso_plan_kg`, `peso_real_kg`, `bultos_plan`, `bultos_real`, `atencion_obj_min`, `prioridad`, `penalidad_hora`, `estado`, `conformidad`, `vol_plan_m3`, `vol_real_m3`, `permite_entrega_tardia`, `tolerancia_tardanza_min`


══════════════════════════════════════════════════════════════════════════════
Puntualidad de descarga respecto del cierre de ventana (sin tolerancia)
══════════════════════════════════════════════════════════════════════════════


,clasificación,entregas
0,A_TIEMPO,1532
1,FUERA_DE_VENTANA,450



══════════════════════════════════════════════════════════════════════════════
Duraciones de descarga negativas o superiores a 8 horas (revisión)
══════════════════════════════════════════════════════════════════════════════


,id_entrega,id_asignacion,duracion_descarga_min
2,20000103,200001,927.0
426,20011001,200110,915.0
629,20015507,200155,957.0
637,20015704,200157,967.0
752,20018501,200185,1189.0
764,20018705,200187,1119.0
863,20021003,200210,998.0
1035,20025304,200253,1033.0
1268,20032101,200321,1239.0
1293,20032905,200329,911.0


**Importante:** este notebook diferencia **desviación frente al plan**, **fin de descarga fuera de la ventana** y **estado de incidencia preexistente**. Son indicadores distintos. Si la auditoría ET utiliza 60 minutos de tolerancia, ese resultado no será idéntico al indicador sin tolerancia de este apartado.

## 7. Recorridos: distancias y consistencia temporal

In [ ]:
display(Markdown('**Campos `recorrido`:** '+', '.join('`'+x+'`' for x in recorrido.columns)))
if 'distancia_km' in recorrido:
 panel('Distancias de recorridos',recorrido['distancia_km'].describe(percentiles=[.1,.25,.5,.75,.9,.95]).to_frame().T.round(2))
 mostrar(px.histogram(recorrido,x='distancia_km',nbins=45,marginal='box',color_discrete_sequence=[PALETTE[0]]),'Distancia por recorrido operacional (km)')
 dist_viaje=recorrido.groupby('id_viaje',as_index=False).agg(km_trayecto=('distancia_km','sum'),tramos_operativos=('id_recorrido','count'))
 mostrar(px.histogram(dist_viaje,x='km_trayecto',nbins=35,color_discrete_sequence=[PALETTE[1]]),'Distancia total de ida por viaje según recorridos D02')
 mostrar(px.histogram(dist_viaje,x='tramos_operativos',nbins=max(10,dist_viaje.tramos_operativos.nunique()),color_discrete_sequence=[PALETTE[2]]),'Número de recorridos operativos por viaje')
else:dist_viaje=pd.DataFrame()
for c in ['entrada_real','salida_real','entrada_plan','salida_plan']:
 if c in recorrido:recorrido[c]=pd.to_datetime(recorrido[c],errors='coerce')
if {'entrada_real','salida_real'}.issubset(recorrido):
 recorrido['duracion_entre_eventos_h']=(recorrido.salida_real-recorrido.entrada_real).dt.total_seconds()/3600
 mostrar(px.histogram(recorrido,x='duracion_entre_eventos_h',nbins=50,color_discrete_sequence=[PALETTE[4]]),'Duración entre entrada y salida del registro recorrido (h)')
 panel('Recorridos con duración negativa',recorrido.loc[recorrido.duracion_entre_eventos_h<0,['id_recorrido','id_viaje','duracion_entre_eventos_h']].head(30))
display(Markdown('⚠️ **`recorrido` representa un desplazamiento operacional, no un segmento vial D03.** Las duraciones entre eventos pueden incluir esperas o descansos; no equivalen automáticamente a horas continuas de conducción.'))


**Campos `recorrido`:** `id_recorrido`, `id_viaje`, `id_tramo`, `secuencia`, `entrada_plan`, `entrada_real`, `salida_plan`, `salida_real`, `distancia_km`, `costo_plan`, `costo_real`, `desvio`, `fuente`


══════════════════════════════════════════════════════════════════════════════
Distancias de recorridos
══════════════════════════════════════════════════════════════════════════════


,count,mean,std,min,10%,25%,50%,75%,90%,95%,max
distancia_km,1982.0,200.97,217.9,4.0,7.0,14.0,132.0,339.0,428.0,560.0,1040.0



══════════════════════════════════════════════════════════════════════════════
Recorridos con duración negativa
══════════════════════════════════════════════════════════════════════════════


,id_recorrido,id_viaje,duracion_entre_eventos_h


⚠️ **`recorrido` representa un desplazamiento operacional, no un segmento vial D03.** Las duraciones entre eventos pueden incluir esperas o descansos; no equivalen automáticamente a horas continuas de conducción.

## 8. Flota, asignaciones y simultaneidad

In [ ]:
usos=viaje.groupby('id_vehiculo',as_index=False).size().rename(columns={'size':'viajes'})
if {'id_vehiculo','placa'}.issubset(vehiculo): usos=usos.merge(vehiculo[['id_vehiculo','placa']],on='id_vehiculo',how='left')
usos['vehiculo_etiqueta']=usos.get('placa',usos['id_vehiculo'].astype(str)).astype(str)
mostrar(px.bar(usos.sort_values('viajes'),x='viajes',y='vehiculo_etiqueta',orientation='h',text='viajes',color_discrete_sequence=[PALETTE[0]]),'Cantidad de viajes asignados por vehículo')
# Intervalos de simultaneidad: inicio servicio y fin de jornada por conductor + intervalo, no por identificador coincidente.
if {'fecha_inicio_eda','id_conductor'}.issubset(viaje) and {'id_conductor','inicio','fin'}.issubset(jornada):
 turnos=jornada.dropna(subset=['inicio','fin'])[['id_conductor','inicio','fin']].copy()
 eventos=[]
 for r in turnos.itertuples(index=False):
  if r.fin>=r.inicio: eventos.extend([(r.inicio,1),(r.fin,-1)])
 eventos=sorted(eventos,key=lambda v:(v[0],v[1]));activos=0;max_activos=0;serie=[]
 for t,v in eventos:
  activos+=v;max_activos=max(activos,max_activos);serie.append((t,activos))
 print('Máxima simultaneidad de jornadas registradas:',max_activos)
 if serie:
  sg=pd.DataFrame(serie,columns=['fecha','activos'])
  mostrar(px.line(sg,x='fecha',y='activos',line_shape='hv',color_discrete_sequence=[PALETTE[1]]),'Jornadas concurrentes (proxy de vehículos ocupados)')
 display(Markdown('**Alcance:** esta simultaneidad se obtiene de los intervalos `jornada_conductor` y es un indicador aproximado de ocupación; **no** demuestra posiciones simultáneas de los camiones.'))


Máxima simultaneidad de jornadas registradas: 9


**Alcance:** esta simultaneidad se obtiene de los intervalos `jornada_conductor` y es un indicador aproximado de ocupación; **no** demuestra posiciones simultáneas de los camiones.

## 9. Costos y jornadas: procedencia y limitaciones

In [ ]:
display(Markdown('**Campos `gasto_viaje`:** '+', '.join('`'+x+'`' for x in gasto.columns)))
if 'importe' in gasto:
 mostrar(px.histogram(gasto,x='importe',nbins=40,color_discrete_sequence=[PALETTE[2]]),'Distribución de importes registrados por gasto')
if 'tipo' in gasto:
 panel('Clasificación de gastos',gasto['tipo'].astype(str).value_counts().head(20).to_frame('registros'))
display(Markdown('**Jornadas:** este bloque usa todos los registros disponibles, no una selección aleatoria. Las estadísticas describen minutos **acumulados** registrados, no intervalos ininterrumpidos de conducción.'))
for c in ['conduccion_acumulada_min','descanso_acumulado_min']:
 if c in jornada:
  z=pd.to_numeric(jornada[c],errors='coerce')
  resumen_j=pd.DataFrame([{'Variable':c,'Válidos':z.notna().sum(),'Mínimo':z.min(),'P25':z.quantile(.25),'Mediana':z.median(),'Promedio':z.mean(),'P75':z.quantile(.75),'P95':z.quantile(.95),'Máximo':z.max()}]).round(1)
  panel('Estadísticos de jornada: '+c+' (min)',resumen_j)
  mostrar(px.histogram(jornada,x=c,nbins=35,marginal='box',color_discrete_sequence=[PALETTE[1]]),f'Jornada: distribución de {c}')
  q1,q3=z.quantile([.25,.75]); limite=q3+1.5*(q3-q1)
  flags=jornada.loc[z>limite,['id_conductor',c]].copy() if 'id_conductor' in jornada else jornada.loc[z>limite,[c]].copy()
  if len(flags):panel(f'Jornadas atípicas de {c} (IQR, límite {limite:,.1f} min; primeros 20)',flags.sort_values(c,ascending=False).head(20))
display(Markdown('**Advertencia:** existen costos por kilómetro, precios de combustible y reglas de conducción explícitamente referenciales o supuestos. No corresponde presentar estas cifras como costos contables o como reglas legales verificadas.'))


**Campos `gasto_viaje`:** `id_gasto`, `id_viaje`, `id_tramo`, `id_parada`, `fecha`, `tipo`, `cantidad`, `unidad`, `precio_unit`, `importe`, `comprobante`, `fuente`


══════════════════════════════════════════════════════════════════════════════
Clasificación de gastos
══════════════════════════════════════════════════════════════════════════════


,registros
tipo,
COSTO_CICLO_BASE_MAS_DEMORA_SIMULADA,500


**Jornadas:** este bloque usa todos los registros disponibles, no una selección aleatoria. Las estadísticas describen minutos **acumulados** registrados, no intervalos ininterrumpidos de conducción.


══════════════════════════════════════════════════════════════════════════════
Estadísticos de jornada: conduccion_acumulada_min (min)
══════════════════════════════════════════════════════════════════════════════


,Variable,Válidos,Mínimo,P25,Mediana,Promedio,P75,P95,Máximo
0,conduccion_acumulada_min,500,777,1114.0,1936.0,1590.0,2046.0,2086.0,2135



══════════════════════════════════════════════════════════════════════════════
Estadísticos de jornada: descanso_acumulado_min (min)
══════════════════════════════════════════════════════════════════════════════


,Variable,Válidos,Mínimo,P25,Mediana,Promedio,P75,P95,Máximo
0,descanso_acumulado_min,500,180,840.0,1680.0,1424.8,2340.0,3000.0,3000


**Advertencia:** existen costos por kilómetro, precios de combustible y reglas de conducción explícitamente referenciales o supuestos. No corresponde presentar estas cifras como costos contables o como reglas legales verificadas.

## 10. Detección automatizada de inconsistencias importantes

In [ ]:
anom=[]
def registrar(nivel,categoria,cantidad,detalle): anom.append({'nivel':nivel,'control':categoria,'cantidad':int(cantidad),'interpretacion':detalle})
if 'ocupacion_pct' in base_viajes:registrar('ALTO','Peso supera capacidad', (base_viajes.ocupacion_pct>100).sum(),'Carga excede capacidad nominal registrada')
if 'duracion_descarga_min' in entrega:
 registrar('ALTO','Descarga con duración negativa',(entrega.duracion_descarga_min<0).sum(),'Fin anterior al inicio de descarga')
 registrar('REVISAR','Descarga > 8 horas',(entrega.duracion_descarga_min>480).sum(),'Puede incluir esperas/pausas o evento incorrecto')
if 'duracion_entre_eventos_h' in recorrido:registrar('ALTO','Recorrido con duración negativa',(recorrido.duracion_entre_eventos_h<0).sum(),'Salida previa a entrada')
if 'cantidad_real' in detalle and 'cantidad_plan' in detalle:
 registrar('REVISAR','Detalle con cantidad real mayor a plan', (pd.to_numeric(detalle.cantidad_real,errors='coerce')>pd.to_numeric(detalle.cantidad_plan,errors='coerce')).sum(),'Validar contra carga transportada; puede existir ajuste legítimo')
for k,df in datos.items():
 if 'id_tramo' in df.columns and k in ['recorrido','gasto_viaje']:
  registrar('PENDIENTE','Sin id_tramo en '+k,df.id_tramo.isna().sum(),'Ruta vial ejecutada no observada; no rellenar como GPS real')
if 'posicion' in datos:registrar('INFORMATIVO','Registros de GPS disponibles',len(datos['posicion']),'No interpretar ausencia de GPS como anomalía de operación')
if auditorias_previas:
 registrar('REVISAR','ET22/ET23 — hojas de auditoría previas',sum(len(d) for d in auditorias_previas.values()),'No integrar estas filas con las tablas operativas vigentes: sus recuentos pueden quedar desactualizados')
anomalias_eda=pd.DataFrame(anom)
panel('Controles priorizados para depuración y metodología',anomalias_eda)



══════════════════════════════════════════════════════════════════════════════
Controles priorizados para depuración y metodología
══════════════════════════════════════════════════════════════════════════════


,nivel,control,cantidad,interpretacion
0,ALTO,Peso supera capacidad,0,Carga excede capacidad nominal registrada
1,ALTO,Descarga con duración negativa,0,Fin anterior al inicio de descarga
2,REVISAR,Descarga > 8 horas,12,Puede incluir esperas/pausas o evento incorrecto
3,ALTO,Recorrido con duración negativa,0,Salida previa a entrada
4,REVISAR,Detalle con cantidad real mayor a plan,0,Validar contra carga transportada; puede exist...
5,PENDIENTE,Sin id_tramo en recorrido,1982,Ruta vial ejecutada no observada; no rellenar ...
6,PENDIENTE,Sin id_tramo en gasto_viaje,500,Ruta vial ejecutada no observada; no rellenar ...
7,INFORMATIVO,Registros de GPS disponibles,0,No interpretar ausencia de GPS como anomalía d...
8,REVISAR,ET22/ET23 — hojas de auditoría previas,5049,No integrar estas filas con las tablas operati...


## 10.1. Laboratorio visual: comparaciones, dispersión y correlaciones

Estos gráficos complementan las distribuciones individuales para observar asociaciones, sesgos y valores atípicos. Una correlación no demuestra causalidad. En los diagramas de caja, los puntos extremos requieren investigación, no eliminación automática.


In [ ]:
# Comparación de carga por vehículo (boxplot) y dispersión entre cajas y peso.
if {'id_vehiculo','total_bultos','peso_total_kg'}.issubset(base_viajes.columns):
    aux=base_viajes.dropna(subset=['id_vehiculo','total_bultos','peso_total_kg']).copy()
    aux['vehiculo']=aux['id_vehiculo'].astype(int).astype(str)
    mostrar(px.box(aux,x='vehiculo',y='total_bultos',points='outliers',color='vehiculo',
                   labels={'vehiculo':'ID de vehículo','total_bultos':'Cajas por viaje'}),
           'Distribución de cajas por vehículo: mediana, dispersión y atípicos')
    mostrar(px.scatter(aux,x='total_bultos',y='peso_total_kg',color='vehiculo',opacity=.7,
                       hover_data=['id_viaje'],labels={'total_bultos':'Cajas','peso_total_kg':'Peso total (kg)'}),
           'Relación entre cantidad de cajas y peso transportado')
else:print('No hay columnas suficientes para los gráficos de carga.')
if 'distancia_km' in recorrido.columns:
    d=pd.to_numeric(recorrido['distancia_km'],errors='coerce').dropna()
    if len(d):
        mostrar(px.box(x=d,points='outliers',labels={'x':'Distancia de cada recorrido (km)'}),
               'Variabilidad y recorridos atípicos por distancia')

num_cols=[c for c in ['total_bultos','peso_total_kg','carga_max_kg','ocupacion_pct'] if c in base_viajes]
if len(num_cols)>=2:
    corr=base_viajes[num_cols].apply(pd.to_numeric,errors='coerce').corr()
    mostrar(px.imshow(corr,text_auto='.2f',zmin=-1,zmax=1,color_continuous_scale='RdBu_r',
                      labels={'color':'Correlación de Pearson'}),
           'Matriz de correlación entre indicadores numéricos de viajes')


## 10.2. Tablero de indicadores para presentación

Indicadores resumidos calculados desde las tablas operativas. La interpretación depende de la completitud y procedencia de los campos.


In [ ]:
from IPython.display import HTML
kpis=[('Viajes',len(viaje)),('Entregas',len(entrega)),('Recorridos',len(recorrido)),('Puntos',len(punto)),('Vehículos',len(vehiculo)),('Cajas promedio por viaje',round(pd.to_numeric(base_viajes.get('total_bultos',pd.Series(dtype=float)),errors='coerce').mean(),1))]
html='<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;font-family:Arial">'
for nombre,valor in kpis:
    html+=f'<div style="border:1px solid #d1d5db;border-radius:12px;padding:16px;background:#f8fafc"><small style="color:#475569">{nombre}</small><h2 style="margin:8px 0;color:#0f172a">{valor:,.1f}' + ('</h2></div>' if isinstance(valor,float) else '</h2></div>')
html+='</div>'
display(HTML(html))
print('Interpretación: los conteos son descriptivos; no prueban por sí solos representatividad de toda la operación.')


Interpretación: los conteos son descriptivos; no prueban por sí solos representatividad de toda la operación.


## 10.3. Análisis relacional integral: de viaje a entrega y punto de atención

**Unidades de análisis independientes:** 1 fila por viaje para métricas de flota; 1 fila por entrega para puntualidad; 1 fila por recorrido para distancias. Las uniones se hacen por `viaje → asignacion → entrega → punto_entrega`, sin multiplicar indebidamente viajes al sumar entregas.

**Definiciones:** `demora_ventana_min = max(fin_descarga - entrega_hasta, 0)`; `desviacion_llegada_min = llegada_real - entrega_objetivo` (puede ser negativa); `tiempo_descarga_min = fin_descarga - inicio_descarga`. Las duraciones representan eventos de una operación reconstruida y no prueban tiempo real de conducción.


In [ ]:
# Vista integrada de entregas, con claves del modelo real verificadas.
rel = (entrega.merge(asignacion[['id_asignacion','id_viaje','id_solicitud']].drop_duplicates('id_asignacion'),on='id_asignacion',how='left',validate='many_to_one')
              .merge(punto[['id_punto','nombre','latitud','longitud']].drop_duplicates('id_punto'),on='id_punto',how='left',validate='many_to_one')
              .merge(viaje[['id_viaje','id_vehiculo','id_conductor','inicio_viaje_real']].drop_duplicates('id_viaje'),on='id_viaje',how='left',validate='many_to_one'))
rel=rel.merge(vehiculo[['id_vehiculo','placa','carga_max_kg']].drop_duplicates('id_vehiculo'),on='id_vehiculo',how='left',validate='many_to_one')
for c in ['entrega_hasta','entrega_desde','entrega_objetivo','llegada_real','inicio_descarga','fin_descarga','inicio_viaje_real']:
 if c in rel:rel[c]=pd.to_datetime(rel[c],errors='coerce')
rel['descarga_min']=(rel.fin_descarga-rel.inicio_descarga).dt.total_seconds()/60
rel['tardanza_min']=((rel.fin_descarga-rel.entrega_hasta).dt.total_seconds()/60).clip(lower=0)
rel['desviacion_llegada_min']=(rel.llegada_real-rel.entrega_objetivo).dt.total_seconds()/60
rel['fuera_ventana']=rel['tardanza_min'].gt(0)
rel['mes_viaje']=rel.inicio_viaje_real.dt.to_period('M').astype(str)
rel['punto_nombre']=rel.id_punto.astype('Int64').astype(str)+' · '+rel.nombre.fillna('Sin nombre').astype(str).str.slice(0,30)
assert len(rel)==len(entrega), 'La relación no debe multiplicar filas de entrega'
print('Vista relacional creada:',len(rel),'entregas; viajes distintos:',rel.id_viaje.nunique())
print('Faltantes en referencia de viaje:',int(rel.id_viaje.isna().sum()),'| Punto:',int(rel.nombre.isna().sum()))
display(rel[['id_viaje','id_entrega','id_punto','nombre','placa','bultos_real','descarga_min','tardanza_min']].head(12))


Vista relacional creada: 1982 entregas; viajes distintos: 500
Faltantes en referencia de viaje: 0 | Punto: 0


,id_viaje,id_entrega,id_punto,nombre,placa,bultos_real,descarga_min,tardanza_min
0,200001,20000101,1,CADENAS 24 HORAS,C0Y781,211,67.0,0.0
1,200001,20000102,2,JIMENES,C0Y781,223,61.0,55.0
2,200001,20000103,20,CADENAS 24 HORAS,C0Y781,331,927.0,909.0
3,200001,20000105,18,NEGOCIOS E INVERSIONES SANTISIMA VIRGEN DE LA ...,C0Y781,150,63.0,0.0
4,200002,20000201,23,PHRYMA,D8W938,256,74.0,0.0
5,200002,20000202,25,DIMEXA,D8W938,248,76.0,916.0
6,200003,20000301,11,PHRYMA,F1S948,316,82.0,0.0
7,200003,20000302,12,REPRESENTACIONES CASTILLO,F1S948,190,89.0,0.0
8,200003,20000303,18,NEGOCIOS E INVERSIONES SANTISIMA VIRGEN DE LA ...,F1S948,354,71.0,911.0
9,200004,20000401,1,CADENAS 24 HORAS,CFN836,119,75.0,0.0


## 10.4. Indicadores estadísticos: mínimo, P25, mediana, promedio, P75, P95 y máximo

La tardanza tiene dos lecturas distintas: **todas las entregas** (incluye ceros) y **solo las entregas tardías**. Esto evita interpretar un promedio general como la demora típica de las entregas incumplidas.


In [ ]:
def estadistica(serie):
 s=pd.to_numeric(serie,errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
 return {'Registros válidos':int(len(s)),'Mínimo':s.min() if len(s) else np.nan,'P25':s.quantile(.25) if len(s) else np.nan,'Mediana':s.median() if len(s) else np.nan,'Promedio':s.mean() if len(s) else np.nan,'P75':s.quantile(.75) if len(s) else np.nan,'P95':s.quantile(.95) if len(s) else np.nan,'Máximo':s.max() if len(s) else np.nan}
rec_por_viaje=recorrido.groupby('id_viaje').agg(km_viaje=('distancia_km','sum'),num_recorridos=('id_recorrido','count'))
ent_por_viaje=rel.groupby('id_viaje').agg(entregas_viaje=('id_entrega','count'),cajas_entregadas=('bultos_real','sum'),tardanza_total_min=('tardanza_min','sum'))
indicadores_detalle={
 'Entregas por viaje (n)':ent_por_viaje.entregas_viaje,
 'Cajas entregadas por viaje (n)':ent_por_viaje.cajas_entregadas,
 'Recorridos por viaje (n)':rec_por_viaje.num_recorridos,
 'Distancia de ida por viaje (km)':rec_por_viaje.km_viaje,
 'Tiempo de descarga por entrega (min)':rel.descarga_min,
 'Tardanza - todas las entregas (min)':rel.tardanza_min,
 'Tardanza - solo tardías (min)':rel.loc[rel.fuera_ventana,'tardanza_min'],
 'Desviación de llegada respecto al objetivo (min)':rel.desviacion_llegada_min,
 'Cajas por entrega (n)':rel.bultos_real,
}
estadisticos=pd.DataFrame({k:estadistica(v) for k,v in indicadores_detalle.items()}).T
panel('TABLA DE ESTADÍSTICAS CONSOLIDADAS',estadisticos.round(2))



══════════════════════════════════════════════════════════════════════════════
TABLA DE ESTADÍSTICAS CONSOLIDADAS
══════════════════════════════════════════════════════════════════════════════


,Registros válidos,Mínimo,P25,Mediana,Promedio,P75,P95,Máximo
Entregas por viaje (n),500.0,1.0,2.00,4.0,3.96,5.0,7.00,9.0
Cajas entregadas por viaje (n),500.0,400.0,871.00,974.5,933.06,1105.0,1246.00,1589.0
Recorridos por viaje (n),500.0,1.0,2.00,4.0,3.96,5.0,7.00,9.0
Distancia de ida por viaje (km),500.0,305.0,571.00,987.0,796.65,1040.0,1075.05,1100.0
Tiempo de descarga por entrega (min),1982.0,60.0,68.00,76.0,87.85,85.0,159.00,1239.0
Tardanza - todas las entregas (min),1982.0,0.0,0.00,0.0,109.01,0.0,922.00,1336.0
Tardanza - solo tardías (min),450.0,1.0,33.50,76.5,480.13,920.0,1030.00,1336.0
Desviación de llegada respecto al objetivo (min),1982.0,-1217.0,-39.25,9.0,-68.36,73.0,261.00,1477.0
Cajas por entrega (n),1982.0,59.0,143.00,202.0,235.38,284.0,517.90,1246.0


### 10.4.1. Estadísticas interpretables y análisis de extremos

Se presentan las métricas por unidad de análisis, con mínimos, medianas, promedios, percentiles y máximos. Los casos extremos **no se eliminan**: se listan para revisión y no se interpreta automáticamente una descarga larga como descarga continua.

In [ ]:
# Estadísticas adicionales por viaje y por punto; conservar cardinalidad de cada unidad.
conteos_viaje=rel.groupby('id_viaje',as_index=False).agg(
    entregas=('id_entrega','count'), puntos_distintos=('id_punto','nunique'),
    cajas=('bultos_real','sum'), tardanza_acumulada_min=('tardanza_min','sum'))
resumen_metricas={
    'Entregas por viaje':conteos_viaje.entregas,
    'Puntos distintos por viaje':conteos_viaje.puntos_distintos,
    'Cajas entregadas por viaje':conteos_viaje.cajas,
    'Tardanza acumulada por viaje (min)':conteos_viaje.tardanza_acumulada_min,
    'Descarga por entrega (min)':rel.descarga_min,
    'Tardanza de entregas tardías (min)':rel.loc[rel.tardanza_min>0,'tardanza_min']
}
resumen_extendido=pd.DataFrame([{'Indicador':k, **estadistica(v)} for k,v in resumen_metricas.items()]).round(2)
panel('RESUMEN ESTADÍSTICO RELACIONAL',resumen_extendido)

# Separar magnitudes con ceros de la distribución condicional > 0
fig=px.box(rel.loc[rel.descarga_min.ge(0)],x='placa',y='descarga_min',points='outliers',
           title='Variabilidad de descarga por camión — valores atípicos visibles')
mostrar(fig)
q1,q3=rel.descarga_min.quantile([.25,.75]); corte=q3+1.5*(q3-q1)
casos_extremos=rel.loc[rel.descarga_min>corte,
    ['id_viaje','id_entrega','punto_nombre','placa','inicio_descarga','fin_descarga','descarga_min']].sort_values('descarga_min',ascending=False)
panel(f'Casos de descarga atípica: {len(casos_extremos)} registros sobre límite IQR {corte:.1f} min (primeros 25)',casos_extremos.head(25))
display(Markdown('**Interpretación:** una duración extrema puede incluir espera, pernocta o interrupciones. El histórico no permite atribuir automáticamente ese intervalo a descarga activa.'))



══════════════════════════════════════════════════════════════════════════════
RESUMEN ESTADÍSTICO RELACIONAL
══════════════════════════════════════════════════════════════════════════════


,Indicador,Registros válidos,Mínimo,P25,Mediana,Promedio,P75,P95,Máximo
0,Entregas por viaje,500,1.0,2.0,4.0,3.96,5.00,7.00,9.0
1,Puntos distintos por viaje,500,1.0,2.0,4.0,3.90,5.00,7.00,9.0
2,Cajas entregadas por viaje,500,400.0,871.0,974.5,933.06,1105.00,1246.00,1589.0
3,Tardanza acumulada por viaje (min),500,0.0,0.0,22.0,432.12,913.25,1942.05,6229.0
4,Descarga por entrega (min),1982,60.0,68.0,76.0,87.85,85.00,159.00,1239.0
5,Tardanza de entregas tardías (min),450,1.0,33.5,76.5,480.13,920.00,1030.00,1336.0



══════════════════════════════════════════════════════════════════════════════
Casos de descarga atípica: 132 registros sobre límite IQR 110.5 min (primeros 25)
══════════════════════════════════════════════════════════════════════════════


,id_viaje,id_entrega,punto_nombre,placa,inicio_descarga,fin_descarga,descarga_min
1268,200321,20032101,17 · CADENAS 24 HORAS,TAZ-855,2026-05-01 11:40:00,2026-05-02 08:19:00,1239.0
752,200185,20018501,17 · CADENAS 24 HORAS,C1O855,2026-03-03 16:19:00,2026-03-04 12:08:00,1189.0
764,200187,20018705,17 · CADENAS 24 HORAS,ARB-752,2026-03-12 14:23:00,2026-03-13 09:02:00,1119.0
1393,200355,20035506,20 · CADENAS 24 HORAS,CFN836,2026-06-07 16:54:00,2026-06-08 10:14:00,1040.0
1035,200253,20025304,17 · CADENAS 24 HORAS,C0Y781,2026-04-11 15:30:00,2026-04-12 08:43:00,1033.0
863,200210,20021003,17 · CADENAS 24 HORAS,F1S948,2026-03-13 16:17:00,2026-03-14 08:55:00,998.0
1930,200485,20048507,17 · CADENAS 24 HORAS,BTG803,2026-07-25 16:12:00,2026-07-26 08:45:00,993.0
637,200157,20015704,20 · CADENAS 24 HORAS,CFN836,2026-02-23 16:41:00,2026-02-24 08:48:00,967.0
629,200155,20015507,17 · CADENAS 24 HORAS,D8W938,2026-02-21 17:22:00,2026-02-22 09:19:00,957.0
2,200001,20000103,20 · CADENAS 24 HORAS,C0Y781,2025-12-08 17:42:00,2025-12-09 09:09:00,927.0


**Interpretación:** una duración extrema puede incluir espera, pernocta o interrupciones. El histórico no permite atribuir automáticamente ese intervalo a descarga activa.

## 10.5. Gráficos relacionales: destinos, puntualidad y descargas

Estas gráficas sirven para explicar **dónde** se concentra el servicio, **cuánto tarda** la descarga y **dónde** aparecen más incumplimientos. La comparación por destino no demuestra por sí sola que un establecimiento cause demoras: las ventanas y escenarios pueden ser distintos.


In [ ]:
res_punto=(rel.groupby(['id_punto','punto_nombre'],dropna=False)
 .agg(entregas=('id_entrega','count'),viajes=('id_viaje','nunique'),cajas=('bultos_real','sum'),
      media_descarga_min=('descarga_min','mean'),promedio_tardanza_min=('tardanza_min','mean'),
      tardias=('fuera_ventana','sum')).reset_index())
res_punto['pct_fuera_ventana']=100*res_punto.tardias/res_punto.entregas
mostrar(px.bar(res_punto.sort_values('entregas'),x='entregas',y='punto_nombre',orientation='h',color='pct_fuera_ventana',color_continuous_scale='YlOrRd',hover_data=['viajes','cajas','media_descarga_min'],height=700), 'Volumen de entregas por destino y porcentaje fuera de ventana')
mostrar(px.scatter(res_punto,x='entregas',y='pct_fuera_ventana',size='cajas',hover_name='punto_nombre',color='media_descarga_min',size_max=45), 'Relación entre demanda, puntualidad y descarga por destino')
mostrar(px.box(rel,x='placa',y='descarga_min',color='placa',points='outliers'), 'Duración de descarga por camión (min)')
mostrar(px.histogram(rel,x='tardanza_min',nbins=45,marginal='box'), 'Distribución de tardanza por entrega, incluyendo entregas puntuales')
mostrar(px.histogram(rel.loc[rel.fuera_ventana],x='tardanza_min',nbins=40,marginal='box'), 'Distribución de tardanza, solo entregas fuera de ventana')


## 10.6. Panel dinámico: filtrar por vehículo y punto

En Colab, usa los selectores para cambiar de vehículo o establecimiento. Las métricas se recalculan sobre **la tabla integrada de entregas**, sin recalcular manualmente el Excel. `Todas` mantiene todos los registros.


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
op_veh=['Todas']+sorted(rel.placa.dropna().astype(str).unique().tolist())
op_punto=['Todos']+sorted(rel.punto_nombre.dropna().astype(str).unique().tolist())
w_veh=widgets.Dropdown(options=op_veh,description='Camión:',layout=widgets.Layout(width='320px'))
w_pun=widgets.Dropdown(options=op_punto,description='Destino:',layout=widgets.Layout(width='570px'))
w_out=widgets.Output()
def actualizar(_=None):
 with w_out:
  clear_output(wait=True)
  x=rel.copy()
  if w_veh.value!='Todas':x=x[x.placa.astype(str)==w_veh.value]
  if w_pun.value!='Todos':x=x[x.punto_nombre.astype(str)==w_pun.value]
  print(f'Entregas: {len(x):,} | Viajes distintos: {x.id_viaje.nunique():,} | Cajas: {pd.to_numeric(x.bultos_real,errors="coerce").sum():,.0f}')
  print(f'Descarga (min): mínimo={x.descarga_min.min():,.1f} | mediana={x.descarga_min.median():,.1f} | promedio={x.descarga_min.mean():,.1f} | máximo={x.descarga_min.max():,.1f}')
  print(f'Fuera de ventana: {x.fuera_ventana.mean()*100:,.1f}% | Tardanza promedio entre tardías={x.loc[x.fuera_ventana,"tardanza_min"].mean():,.1f} min')
  if len(x):
   fig=px.histogram(x,x='descarga_min',nbins=30,marginal='box',title='Descargas según filtro actual')
   fig.update_layout(height=350);fig.show()
   fig=px.histogram(x,x='tardanza_min',nbins=30,title='Tardanzas según filtro actual')
   fig.update_layout(height=350);fig.show()
w_veh.observe(actualizar,names='value');w_pun.observe(actualizar,names='value')
display(widgets.HBox([w_veh,w_pun]),w_out);actualizar()


Output()

## 10.7. Comparación de entregas planificadas vs ejecutadas e integridad operacional

Los contrastes de cantidades y horarios deben interpretarse de acuerdo con el origen del dato (observado, reconstruido o sintético). Los valores fuera de ventana son hallazgos operativos plausibles, **no errores que deban borrarse**.


In [ ]:
comp=rel[['id_entrega','id_viaje','punto_nombre','bultos_plan','bultos_real','peso_plan_kg','peso_real_kg','tardanza_min']].copy()
comp['diferencia_bultos']=pd.to_numeric(comp.bultos_real,errors='coerce')-pd.to_numeric(comp.bultos_plan,errors='coerce')
comp['diferencia_peso_kg']=pd.to_numeric(comp.peso_real_kg,errors='coerce')-pd.to_numeric(comp.peso_plan_kg,errors='coerce')
mostrar(px.scatter(comp,x='bultos_plan',y='bultos_real',color='tardanza_min',hover_data=['id_entrega','id_viaje']), 'Cantidad planificada frente a entregada')
panel('Resumen de diferencias plan / real',comp[['diferencia_bultos','diferencia_peso_kg']].describe().T.round(2))
print('Entregas sin viaje asociado:',int(rel.id_viaje.isna().sum()))
print('Entregas sin punto asociado:',int(rel.nombre.isna().sum()))
print('Descargas con duración negativa:',int(rel.descarga_min.lt(0).sum()))
print('Entregas donde lo ejecutado supera lo planificado:',int(comp.diferencia_bultos.gt(0).sum()))
print('Recorridos D02 sin id_tramo geográfico:',int(recorrido.id_tramo.isna().sum()) if 'id_tramo' in recorrido else 'Campo ausente')



══════════════════════════════════════════════════════════════════════════════
Resumen de diferencias plan / real
══════════════════════════════════════════════════════════════════════════════


,count,mean,std,min,25%,50%,75%,max
diferencia_bultos,1982.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
diferencia_peso_kg,1982.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Entregas sin viaje asociado: 0
Entregas sin punto asociado: 0
Descargas con duración negativa: 0
Entregas donde lo ejecutado supera lo planificado: 0
Recorridos D02 sin id_tramo geográfico: 1982


## 11. Resumen interpretativo listo para presentar

In [ ]:
n_v=len(viaje);n_e=len(entrega);n_r=len(recorrido)
pct_tarde=100*(entrega.get('puntual_ventana_eda',pd.Series(dtype=str))=='FUERA_DE_VENTANA').mean() if len(entrega) else np.nan
media_cajas=base_viajes.total_bultos.mean() if 'total_bultos' in base_viajes else np.nan
media_km=dist_viaje.km_trayecto.mean() if not dist_viaje.empty else np.nan
texto=f"""### Principales hallazgos del archivo analizado

- Se identificaron **{n_v:,} viajes**, **{n_e:,} entregas**, **{n_r:,} recorridos operativos** y **{len(punto):,} puntos de entrega**.
- La demanda promedio registrada es de **{media_cajas:,.1f} bultos/cajas por viaje**.
- La distancia media de recorridos sumados por viaje es de **{media_km:,.1f} km** (sin incluir necesariamente el retorno vacío).
- **{pct_tarde:,.1f}%** de las entregas tienen fin de descarga posterior al cierre de ventana registrado, **sin tolerancia**.
- El Excel no contiene trayectorias GPS en la hoja `posicion`; los registros `recorrido` no indican qué carreteras individuales fueron transitadas.
- Las hojas `ET22_*` y `ET23_*` se deben tratar como artefactos de auditorías anteriores, no como fuentes vigentes para contar entregas.

**Interpretación:** esta exploración permite describir el entorno y orientar la calibración de un simulador MARL, pero no demuestra aún eficacia del aprendizaje ni exactitud de reconstrucción vial. Debe completarse con procedencia de datos y evaluación experimental independiente.
"""
display(Markdown(texto))


### Principales hallazgos del archivo analizado

- Se identificaron **500 viajes**, **1,982 entregas**, **1,982 recorridos operativos** y **25 puntos de entrega**.
- La demanda promedio registrada es de **933.1 bultos/cajas por viaje**.
- La distancia media de recorridos sumados por viaje es de **796.7 km** (sin incluir necesariamente el retorno vacío).
- **22.7%** de las entregas tienen fin de descarga posterior al cierre de ventana registrado, **sin tolerancia**.
- El Excel no contiene trayectorias GPS en la hoja `posicion`; los registros `recorrido` no indican qué carreteras individuales fueron transitadas.
- Las hojas `ET22_*` y `ET23_*` se deben tratar como artefactos de auditorías anteriores, no como fuentes vigentes para contar entregas.

**Interpretación:** esta exploración permite describir el entorno y orientar la calibración de un simulador MARL, pero no demuestra aún eficacia del aprendizaje ni exactitud de reconstrucción vial. Debe completarse con procedencia de datos y evaluación experimental independiente.


## 12. Exportación de resultados (HTML visual + Excel de controles)

In [ ]:
from plotly.io import to_html
from html import escape
html_parts=['<!doctype html><html lang="es"><head><meta charset="utf-8"><title>EDA EYM D01 D02</title><style>body{font-family:Arial,sans-serif;margin:30px auto;max-width:1200px;color:#13213d}h1{color:#153b79}.note{background:#edf4fc;border-left:4px solid #2463bf;padding:14px}</style></head><body><h1>EDA — EYM D01/D02 V5</h1><div class="note">Resultados exploratorios del archivo cargado. Datos reconstruidos/sintéticos, no todos comprobados operacionalmente. ET22/ET23 excluidas de conteos vigentes.</div>']
for idx,fig in enumerate(PLOTLY_FIGS):
 try: html_parts.append(to_html(fig,include_plotlyjs=('cdn' if idx==0 else False),full_html=False))
 except Exception as exc: print('Figura omitida:',exc)
html_parts.append('</body></html>')
reporte='/content/EDA_EYM_D01_D02_V5.html'
with open(reporte,'w',encoding='utf8') as f:f.write('\n'.join(html_parts))
controles_path='/content/EDA_EYM_Controles.xlsx'
with pd.ExcelWriter(controles_path,engine='xlsxwriter') as writer:
 resumen_hojas.to_excel(writer,sheet_name='Inventario',index=False)
 calidad.to_excel(writer,sheet_name='Completitud',index=False)
 pd.DataFrame(controles).to_excel(writer,sheet_name='Claves_Primarias',index=False)
 pd.DataFrame(resultados_fk).to_excel(writer,sheet_name='Relaciones_FK',index=False)
 anomalias_eda.to_excel(writer,sheet_name='Observaciones',index=False)
 base_viajes.to_excel(writer,sheet_name='Resumen_Viajes',index=False)
 if not dest.empty: dest.to_excel(writer,sheet_name='Destinos',index=False)
print('Informe HTML:',reporte,'| Excel de controles:',controles_path)
from google.colab import files
files.download(reporte)
files.download(controles_path)


Informe HTML: /content/EDA_EYM_D01_D02_V5.html | Excel de controles: /content/EDA_EYM_Controles.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Guía para interpretar este EDA en la sustentación

**Descriptivo:** cuántos viajes, dónde entregamos, cuántas cajas movilizamos, qué proporción de ventanas se excede y qué tan simultánea es la operación.

**Calidad:** qué campos están vacíos, si hay referencias huérfanas, temporalidades imposibles, excedentes de capacidad o duplicación de identificadores.

**Implicaciones para MARL:** el conjunto puede alimentar escenarios de planificación, cargas, tiempos y ventanas; para rutas efectivas se necesita un motor sobre red vial y para tráfico se requieren supuestos calibrados o datos externos. **No tenemos posiciones GPS históricas ni la ruta exacta ejecutada**.

**Antes de publicar:** distinguir fuente observada, reconstruida y sintética; actualizar ET22/ET23 y documentar las reglas de tolerancia y las limitaciones del simulador.

## 13. Descarga opcional de CSV individuales

Si deseas trabajar en otro proyecto, descarga el ZIP y carga las entidades con `pd.read_csv('viaje.csv')`. Los CSV son una representación interoperable del Excel; conserva el archivo XLSX como fuente y registra versión y fecha de conversión.


In [ ]:
# Ejecutar en Google Colab para guardar la exportación CSV completa.
from google.colab import files
files.download(RUTA_ZIP)


## 14. Hallazgos relacionales y su utilidad para MARL

- **Demanda y carga:** distribución de cajas, peso y capacidad → variables de observación y restricciones.
- **Entregas por punto:** concentración geográfica → escenarios de destinos y prioridades.
- **Descarga y tardanza:** estadísticas, percentiles y colas → calibración de atención y penalizaciones.
- **Simultaneidad:** jornadas solapadas → escenarios multiagente, no prueba de ubicación real.
- **Rutas:** `recorrido` registra trayectos operativos, **no** tramos OSM ejecutados; por ello D03/D04 requieren un motor externo si se modela selección de rutas.
- **Sesgos y procedencia:** separar registros reales de datos reconstruidos para la validación experimental.

**Para la presentación:** seleccionar gráficos que respondan a una pregunta, explicar tamaño de muestra y unidad de análisis y cerrar con limitaciones y decisiones de diseño MARL.
